[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/22_robotics_action_policies.ipynb)

# 22. Robotics action policies

BC에서 action chunking, diffusion action step, flow matching action, autoregressive action token으로 진행한다.

**반복 형식:** 바닐라 PyTorch 실행 → profiler로 ATen/CUDA 연산 확인 → 필요할 때만 작은 텐서로 수학적 전개를 펼친다.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)


In [ ]:
from torch.profiler import profile, ProfilerActivity

def profile_call(name, fn, *args, **kwargs):
    activities = [ProfilerActivity.CPU]
    if torch.cuda.is_available():
        activities.append(ProfilerActivity.CUDA)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    with profile(
        activities=activities,
        record_shapes=True,
        profile_memory=True,
        with_stack=False,
    ) as prof:
        out = fn(*args, **kwargs)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    print(f"\n[{name}] top operators")
    sort_key = "self_cuda_time_total" if torch.cuda.is_available() else "self_cpu_time_total"
    print(prof.key_averages().table(sort_by=sort_key, row_limit=12))

    return out


## 1. Behavior cloning regression

observation에서 continuous action을 직접 회귀한다.


In [ ]:
obs = torch.tensor([[1.,0.,-1.,2.]], device=device)
target_action = torch.tensor([[0.2,-0.4]], device=device)

policy = nn.Linear(4,2).to(device)
pred = policy(obs)
loss = F.mse_loss(pred,target_action)

print("pred:", pred)
print("BC loss:", loss.item())


In [ ]:
_ = profile_call("BC forward", policy, obs)


## 2. Action chunking

한 시점에서 여러 미래 action을 한 번에 출력한다.


In [ ]:
chunk = 3
action_dim = 2
chunk_head = nn.Linear(4, chunk*action_dim).to(device)

action_chunk = chunk_head(obs).view(1,chunk,action_dim)
print(action_chunk)


In [ ]:
_ = profile_call("action chunk head", lambda z: chunk_head(z).view(1,chunk,action_dim), obs)


## 3. ACT-style latent-conditioned chunk

observation latent와 chunk position embedding을 결합한다.


In [ ]:
d = 8
obs_proj = nn.Linear(4,d).to(device)
pos = nn.Embedding(chunk,d).to(device)
out = nn.Linear(d,action_dim).to(device)

h = obs_proj(obs)[:,None] + pos(torch.arange(chunk,device=device))[None]
act = out(F.silu(h))
print(act.shape)


In [ ]:
def act_style_chunk_once():
    obs_ = obs_proj(obs)[:, None]
    pos_ = pos(torch.arange(chunk, device=device))[None]
    return out(F.silu(obs_ + pos_))

_ = profile_call("ACT-style chunk", act_style_chunk_once)


## 4. Diffusion action denoising step

noisy action을 epsilon prediction으로 한 step 복원한다.


In [ ]:
a_t = torch.tensor([[0.8,-0.5]], device=device)
pred_eps = torch.tensor([[0.3,-0.1]], device=device)
alpha = torch.tensor(0.9,device=device)

a0_hat = (a_t - torch.sqrt(1-alpha**2)*pred_eps) / alpha
print("estimated clean action:", a0_hat)


In [ ]:
_ = profile_call("action denoise", lambda: (a_t-torch.sqrt(1-alpha**2)*pred_eps)/alpha)


## 5. Flow matching action target

noise action과 expert action 사이 velocity를 학습 target으로 둔다.


In [ ]:
noise_action = torch.tensor([[-1.,1.]], device=device)
expert_action = torch.tensor([[0.2,-0.4]], device=device)
t = torch.tensor([[0.3]], device=device)

a_t = (1-t)*noise_action + t*expert_action
target_v = expert_action-noise_action

print("a_t:", a_t)
print("target velocity:", target_v)


In [ ]:
_ = profile_call("flow action target", lambda: ((1-t)*noise_action+t*expert_action,expert_action-noise_action))


## 6. Flow policy sampling

예측 velocity뉼 몇 step 적분해 continuous action을 만든다.


In [ ]:
action = noise_action.clone()

def tiny_velocity(a, t):
    return 0.5 * (expert_action - a)

for i in range(4):
    t = torch.tensor([[i/4]],device=device)
    action = action + 0.25*tiny_velocity(action,t)
    print(i,action)


In [ ]:
def one_flow_policy_step(a):
    t_ = torch.tensor([[0.0]], device=device)
    return a + 0.25 * tiny_velocity(a, t_)

_ = profile_call("one flow policy step", one_flow_policy_step, noise_action)


## 7. Autoregressive action tokens

continuous action을 bin으로 양자화해 token classification으로 바꾼다.


In [ ]:
bins = torch.linspace(-1,1,9,device=device)
action = torch.tensor([[-0.4,0.7]],device=device)
token_ids = torch.bucketize(action,bins)-1
token_ids = token_ids.clamp(0,len(bins)-2)

print("action:",action)
print("token ids:",token_ids)


In [ ]:
_ = profile_call("action tokenization", lambda: (torch.bucketize(action,bins)-1).clamp(0,len(bins)-2))


## References and provenance

**[22.1] Behavior Cloning**
- 출처: standard imitation learning
- 이 노트북에서 가져온 부분: continuous action regression

**[22.2] ACT**
- 출처: Zhao et al., Learning Fine-Grained Bimanual Manipulation with Low-Cost Hardware
- 이 노트북에서 가져온 부분: action chunking with transformer

**[22.3] Diffusion Policy**
- 출처: Chi et al., Diffusion Policy
- 이 노트북에서 가져온 부분: iterative denoising in action space

**[22.4] π0 / openpi**
- 출처: Physical Intelligence π0 family
- 이 노트북에서 가져온 부분: flow-matching continuous action head

**[22.5] π0-FAST / autoregressive VLA lineages**
- 출처: Physical Intelligence FAST/action-token work
- 이 노트북에서 가져온 부분: continuous action tokenization and autoregressive decoding
